In [ ]:
# ==========================
# LANGUAGE DETECTION BACKEND
# Google Colab Version
# ==========================

!pip install scikit-learn pandas numpy joblib -q

import pandas as pd
import numpy as np
import re
import joblib

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support
)

# ===================================
# STEP 1 : Upload CSV
# ===================================

print("Upload Language_Detection.csv")

uploaded = files.upload()

csv_name = list(uploaded.keys())[0]

df = pd.read_csv(csv_name)

print("\nDataset Loaded Successfully")
print(df.head())

# ===================================
# STEP 2 : Data Cleaning
# ===================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df = df.dropna()

df = df.drop_duplicates(subset=["Text"])

df["clean_text"] = df["Text"].apply(clean_text)

print("\nTotal Records :", len(df))

# ===================================
# STEP 3 : Encode Labels
# ===================================

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["Language"])

# ===================================
# STEP 4 : Train Test Split
# ===================================

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ===================================
# STEP 5 : TFIDF Vectorization
# ===================================

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2,5),
    max_features=10000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_text)

X_test = vectorizer.transform(X_test_text)

print("\nVectorization Complete")

# ===================================
# STEP 6 : Models
# ===================================

svm_model = LinearSVC(
    C=2.0,
    max_iter=10000
)

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

# ===================================
# STEP 7 : Train SVM
# ===================================

print("\nTraining SVM...")

svm_model.fit(X_train,y_train)

svm_pred = svm_model.predict(X_test)

svm_acc = accuracy_score(y_test,svm_pred)

print("\nSVM Accuracy :",round(svm_acc*100,2),"%")

# ===================================
# STEP 8 : Train KNN
# ===================================

print("\nTraining KNN...")

knn_model.fit(X_train,y_train)

knn_pred = knn_model.predict(X_test)

knn_acc = accuracy_score(y_test,knn_pred)

print("\nKNN Accuracy :",round(knn_acc*100,2),"%")

# ===================================
# STEP 9 : KMeans (Unsupervised)
# ===================================

print("\nTraining KMeans...")

kmeans_model = KMeans(
    n_clusters=df["Language"].nunique(),
    random_state=42,
    n_init=20
)

kmeans_model.fit(X_train)

print("KMeans Training Complete")

# ===================================
# STEP 10 : Best Model
# ===================================

if svm_acc > knn_acc:

    best_model = svm_model
    best_name = "Linear SVM"

else:

    best_model = knn_model
    best_name = "KNN"

print("\nBest Model :",best_name)

# ===================================
# STEP 11 : Save Models
# ===================================

joblib.dump(best_model,"language_model.pkl")
joblib.dump(vectorizer,"vectorizer.pkl")
joblib.dump(encoder,"label_encoder.pkl")

print("\nFiles Saved Successfully")

# ===================================
# STEP 12 : Prediction Function
# ===================================

def predict_language(text):

    if len(text.split()) < 2:
        return "Please enter at least 2-3 words."

    text = clean_text(text)

    vec = vectorizer.transform([text])

    pred = best_model.predict(vec)[0]

    lang = encoder.inverse_transform([pred])[0]

    return lang

# ===================================
# STEP 13 : Test Prediction
# ===================================

while True:

    text = input("\nEnter Text : ")

    if text.lower() == "exit":
        break

    print("Predicted Language :",predict_language(text))

Upload Language_Detection.csv


Saving Language_Detection.csv to Language_Detection.csv

Dataset Loaded Successfully
                                                Text Language
0   Nature, in the broadest sense, is the natural...  English
1  "Nature" can refer to the phenomena of the phy...  English
2  The study of nature is a large, if not the onl...  English
3  Although humans are part of nature, human acti...  English
4  [1] The word nature is borrowed from the Old F...  English

Total Records : 10267

Vectorization Complete

Training SVM...

SVM Accuracy : 98.98 %

Training KNN...

KNN Accuracy : 96.79 %

Training KMeans...
